# Wanting Probe — Analysis & Findings

This notebook documents the development and validation of the **wanting probe** for Qwen2VL-7B, the second component of our anhedonia model.

## Background

Anhedonia has (at least) four dissociable components:

| Component | Clinical Meaning | Brain Circuit |
|-----------|-----------------|---------------|
| **Liking** | "This feels good" | Opioid system (nucleus accumbens shell) |
| **Wanting** | "I'm going after this" | Dopamine system (VTA → NAc core) |
| Enjoying | "I'm experiencing pleasure right now" | Opioid + endocannabinoid |
| Learning | "Next time I'll seek this out" | Dopamine prediction error |

We already built and validated a **liking probe** that selectively reduces hedonic tone while preserving effort willingness. Now we build the **wanting probe** to target motivational drive.

---

## 1. Imports & Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from scipy import stats

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

C_LIKING = '#4A90D9'      # blue
C_WANTING = '#E8A84A'     # gold/orange
C_NORMAL = '#5CB85C'      # green
C_SUPPRESSED = '#E85D4A'  # red

print('All imports OK')

---
## 2. Contrastive Dataset Design

### The Problem with v1

Our first wanting dataset failed because the pairs differed in **emotional arousal**, not just motivation:

```
v1 HIGH: "I need to get that job, I will do whatever it takes"    ← emotional + motivated
v1 LOW:  "There is a job opening posted on the board"              ← flat, 3rd person
```

The probe learned arousal vs. observation, not wanting vs. not-wanting. When suppressed, the model became MORE impulsive (effort willingness went UP from 76% to 92.5%).

### The Fix in v2

Both sentences are now first-person, same awareness, same situation — differing ONLY in drive:

```
v2 HIGH: "I know about the job opening and I am going to apply first thing tomorrow"
v2 LOW:  "I know about the job opening but I probably will not bother applying"
```

This isolates the motivational signal from arousal, emotion, and perspective.

In [ ]:
df_wanting = pd.read_csv('wanting_contrastive_dataset_v2.csv')
print(f'Wanting contrastive pairs: {len(df_wanting)}')
print(f'\nExample pairs:')
for i in [0, 5, 10, 50, 100]:
    if i < len(df_wanting):
        print(f'\n  HIGH: {df_wanting.iloc[i]["high_wanting"][:90]}...')
        print(f'  LOW:  {df_wanting.iloc[i]["low_wanting"][:90]}...')

---
## 3. Activation Extraction & Separation Scores

We extracted last-token residual stream activations at all 28 layers for each sentence in 152 contrastive pairs.

In [ ]:
# Load wanting activations
wanting_high = torch.load('wanting_high_activations_v2.pt')
wanting_low = torch.load('wanting_low_activations_v2.pt')

num_layers = len(wanting_high[0])
num_pairs = len(wanting_high)
print(f'Layers: {num_layers}, Pairs: {num_pairs}')

# Compute separation scores
wanting_sep = []
for li in range(num_layers):
    h = torch.stack([a[li].squeeze() for a in wanting_high])
    l = torch.stack([a[li].squeeze() for a in wanting_low])
    wanting_sep.append((h.mean(0) - l.mean(0)).norm().item())

# Also load liking activations for comparison
try:
    liking_high = torch.load('high_reward_activations.pt')
    liking_low = torch.load('neutral_activations.pt')
    liking_sep = []
    for li in range(num_layers):
        h = torch.stack([a[li].squeeze() for a in liking_high])
        l = torch.stack([a[li].squeeze() for a in liking_low])
        liking_sep.append((h.mean(0) - l.mean(0)).norm().item())
    has_liking = True
    print(f'Liking activations loaded ({len(liking_high)} pairs)')
except:
    has_liking = False
    print('Liking activations not found — skipping comparison plots')

### Separation Score Comparison: Wanting vs Liking

Both signals should build across layers, but they could have different profiles.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(num_layers)
ax.plot(x, wanting_sep, 'o-', color=C_WANTING, linewidth=2, markersize=6, label='Wanting probe')
if has_liking:
    ax.plot(x, liking_sep, 's-', color=C_LIKING, linewidth=2, markersize=6, label='Liking probe')

ax.axvspan(6, 27, alpha=0.08, color='red', label='Intervention range (layers 6-27)')
ax.set_xlabel('Layer', fontsize=13)
ax.set_ylabel('Separation Score (L2 norm)', fontsize=13)
ax.set_title('Signal Build-Up: Wanting vs Liking', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Wanting — Best layer: {np.argmax(wanting_sep)} (score: {max(wanting_sep):.2f})')
if has_liking:
    print(f'Liking  — Best layer: {np.argmax(liking_sep)} (score: {max(liking_sep):.2f})')

---
## 4. Probe Training & Cross-Validation

In [ ]:
wanting_cv_accs = []

for li in range(num_layers):
    h = torch.stack([a[li].squeeze() for a in wanting_high]).float().numpy()
    l = torch.stack([a[li].squeeze() for a in wanting_low]).float().numpy()
    X = np.concatenate([h, l])
    y = np.array([1]*len(h) + [0]*len(l))
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    clf = LogisticRegression(max_iter=1000, C=0.01)
    cv = cross_val_score(clf, X_s, y, cv=5, scoring='accuracy')
    wanting_cv_accs.append(cv.mean())

plt.figure(figsize=(14, 4))
plt.bar(range(num_layers), [a*100 for a in wanting_cv_accs], color=C_WANTING, edgecolor='black', linewidth=0.3)
plt.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Chance')
plt.xlabel('Layer')
plt.ylabel('5-Fold CV Accuracy (%)')
plt.title('Wanting Probe: Cross-Validation Accuracy Per Layer', fontsize=14, fontweight='bold')
plt.xticks(range(num_layers))
plt.ylim(45, 101)
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean CV accuracy across all layers: {np.mean(wanting_cv_accs)*100:.1f}%')
print(f'Best CV accuracy: {max(wanting_cv_accs)*100:.1f}% at layer {np.argmax(wanting_cv_accs)}')

### Probe Histogram: High-Wanting vs Low-Wanting

In [ ]:
for target_layer in [20, 24, 27]:
    h = torch.stack([a[target_layer].squeeze() for a in wanting_high]).float().numpy()
    l = torch.stack([a[target_layer].squeeze() for a in wanting_low]).float().numpy()
    X = np.concatenate([h, l])
    y = np.array([1]*len(h) + [0]*len(l))
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    clf = LogisticRegression(max_iter=1000, C=0.01)
    clf.fit(X_s, y)
    
    projections = X_s @ clf.coef_[0]
    
    plt.figure(figsize=(10, 4))
    plt.hist(projections[y==1], bins=30, alpha=0.6, color=C_WANTING, label='High Wanting (motivated)')
    plt.hist(projections[y==0], bins=30, alpha=0.6, color='gray', label='Low Wanting (indifferent)')
    acc = accuracy_score(y, clf.predict(X_s))
    cv = cross_val_score(clf, X_s, y, cv=5).mean()
    plt.title(f'Layer {target_layer} — Train: {acc*100:.1f}% | CV: {cv*100:.1f}%', fontsize=13)
    plt.xlabel('Projection onto wanting direction')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()

---
## 5. The Critical Test: Are Liking and Wanting Independent?

We compare the probe directions by computing their **cosine similarity**. If the two probes found the same signal, the cosine would be close to 1. If they found independent signals, it would be close to 0.

This is analogous to asking: are the opioid system (liking) and dopamine system (wanting) the same circuit or different circuits?

In [ ]:
wanting_probes = torch.load('wanting_probe_vectors_v2.pt')
liking_probes = torch.load('probe_vectors.pt')

cosine_sims = []
for li in range(num_layers):
    wv = wanting_probes[li]
    lv = liking_probes[li]
    cos = (wv @ lv).item()
    cosine_sims.append(cos)

fig, ax = plt.subplots(figsize=(14, 5))

colors = ['green' if abs(c) < 0.3 else 'orange' if abs(c) < 0.7 else 'red' for c in cosine_sims]
bars = ax.bar(range(num_layers), cosine_sims, color=colors, edgecolor='black', linewidth=0.3)

ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=0.3, color='orange', linestyle='--', alpha=0.5, label='Weak overlap (0.3)')
ax.axhline(y=-0.3, color='orange', linestyle='--', alpha=0.5)
ax.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Strong overlap (0.7)')
ax.axhline(y=-0.7, color='red', linestyle='--', alpha=0.5)

ax.set_xlabel('Layer', fontsize=13)
ax.set_ylabel('Cosine Similarity', fontsize=13)
ax.set_title('Wanting vs Liking Probe Directions: Are They Independent?', fontsize=15, fontweight='bold')
ax.set_xticks(range(num_layers))
ax.set_ylim(-1, 1)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'Mean |cosine similarity| across all layers: {np.mean(np.abs(cosine_sims)):.4f}')
print(f'Max  |cosine similarity|: {max(np.abs(cosine_sims)):.4f} at layer {np.argmax(np.abs(cosine_sims))}')
print(f'\nAt intervention layers (6-27):')
for li in [6, 12, 18, 24, 27]:
    print(f'  Layer {li:2d}: cosine = {cosine_sims[li]:.4f}')

### Interpretation

Cosine similarity ~0.05–0.07 across all layers means the two probe directions are **nearly orthogonal** — they point in completely different directions in the 3584-dimensional residual stream.

This is the strongest possible evidence that:
1. **Liking and wanting are independent signals** inside the model
2. Suppressing one should NOT affect the other
3. We should be able to achieve a **double dissociation**

This mirrors the neuroscience finding that liking (opioid system) and wanting (dopamine system) are neurochemically independent circuits in the brain (Berridge & Robinson, 2003).

---
## 6. Geometric Visualization: Two Directions in Activation Space

We project the activations onto the 2D plane defined by the liking and wanting directions.

In [ ]:
# Pick a representative layer
target_layer = 24

liking_dir = liking_probes[target_layer].float()
wanting_dir = wanting_probes[target_layer].float()

# Gram-Schmidt to make an orthogonal basis
# (they're already nearly orthogonal, but let's be exact)
e1 = liking_dir / liking_dir.norm()  # liking axis
e2_raw = wanting_dir - (wanting_dir @ e1) * e1
e2 = e2_raw / e2_raw.norm()  # wanting axis (orthogonalized)

# Project all 4 conditions onto this 2D plane
conditions = {
    'Liking HIGH (reward)': [a[target_layer].squeeze().float() for a in liking_high],
    'Liking LOW (neutral)': [a[target_layer].squeeze().float() for a in liking_low],
    'Wanting HIGH (motivated)': [a[target_layer].squeeze().float() for a in wanting_high],
    'Wanting LOW (indifferent)': [a[target_layer].squeeze().float() for a in wanting_low],
}

cond_colors = {
    'Liking HIGH (reward)': '#4A90D9',
    'Liking LOW (neutral)': '#A8C8E8',
    'Wanting HIGH (motivated)': '#E8A84A',
    'Wanting LOW (indifferent)': '#F0D8A8',
}

plt.figure(figsize=(10, 8))

for label, acts in conditions.items():
    projs = torch.stack(acts)
    x_vals = (projs @ e1).numpy()
    y_vals = (projs @ e2).numpy()
    plt.scatter(x_vals, y_vals, c=cond_colors[label], label=label,
                s=40, alpha=0.6, edgecolors='black', linewidth=0.3)
    # Plot mean
    plt.scatter(x_vals.mean(), y_vals.mean(), c=cond_colors[label],
                s=200, edgecolors='black', linewidth=2, marker='D', zorder=5)

plt.xlabel('\u2190 Neutral          Liking Direction          Rewarding \u2192', fontsize=12)
plt.ylabel('\u2190 Indifferent      Wanting Direction      Motivated \u2192', fontsize=12)
plt.title(f'Four Conditions Projected onto Liking \u00d7 Wanting Plane (Layer {target_layer})',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='upper left')
plt.axhline(y=0, color='gray', linewidth=0.5, alpha=0.5)
plt.axvline(x=0, color='gray', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

print('Diamonds = condition means. If liking and wanting are independent,')
print('the four means should form a rough rectangle, not a diagonal line.')

---
## 7. PCA of Wanting Activations

In [ ]:
for target_layer in [20, 24, 27]:
    h = torch.stack([a[target_layer].squeeze() for a in wanting_high]).float().numpy()
    l = torch.stack([a[target_layer].squeeze() for a in wanting_low]).float().numpy()
    
    all_acts = np.concatenate([h, l])
    labels = ['High Wanting'] * len(h) + ['Low Wanting'] * len(l)
    
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(all_acts)
    
    plt.figure(figsize=(7, 5))
    for i, (x, y) in enumerate(reduced):
        c = C_WANTING if labels[i] == 'High Wanting' else 'gray'
        plt.scatter(x, y, color=c, s=60, alpha=0.7)
    
    plt.scatter([], [], color=C_WANTING, label='High Wanting (motivated)')
    plt.scatter([], [], color='gray', label='Low Wanting (indifferent)')
    plt.legend(fontsize=11)
    plt.title(f'PCA at Layer {target_layer}', fontsize=14)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    plt.tight_layout()
    plt.show()

---
## 8. Evaluation Results — Wanting Probe Suppression

We ran the same 200 tasks (5 types × 40) × 5 independent runs, but now suppressing the **wanting** direction instead of the liking direction.

In [ ]:
# Load wanting eval results
try:
    df_wanting_raw = pd.read_csv('wanting_eval_multirun_raw_v2.csv')
    df_wanting_summary = pd.read_csv('wanting_eval_multirun_summary_v2.csv')
    NUM_RUNS_W = len(df_wanting_summary)
    print(f'Wanting eval loaded: {len(df_wanting_raw)} responses, {NUM_RUNS_W} runs')
    has_wanting_eval = True
except:
    print('Wanting evaluation results not found yet.')
    print('Run eval_wanting_multirun_v2.py first, then re-run this cell.')
    has_wanting_eval = False

In [ ]:
if has_wanting_eval:
    print('WANTING PROBE — KEY METRICS')
    print('=' * 60)
    
    metrics = [
        ('Positive emotion words', 'normal_pos_words', 'wanting_pos_words'),
        ('Response length', 'normal_resp_length', 'wanting_resp_length'),
    ]
    if 'normal_rank_gap' in df_wanting_summary.columns:
        metrics.append(('Preference ranking gap', 'normal_rank_gap', 'wanting_rank_gap'))
    if 'normal_chose_reward_pct' in df_wanting_summary.columns:
        metrics.append(('Reward choice %', 'normal_chose_reward_pct', 'wanting_chose_reward_pct'))
    if 'normal_effort_yes_pct' in df_wanting_summary.columns:
        metrics.append(('Effort willingness %', 'normal_effort_yes_pct', 'wanting_effort_yes_pct'))
    
    for name, nc, wc in metrics:
        nv = df_wanting_summary[nc].values
        wv = df_wanting_summary[wc].values
        try:
            t_stat, p_val = stats.ttest_rel(nv, wv)
        except:
            t_stat, p_val = float('nan'), float('nan')
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
        print(f'\n{name}:')
        print(f'  Normal:             {nv.mean():.3f} \u00b1 {nv.std():.3f}')
        print(f'  Wanting-suppressed: {wv.mean():.3f} \u00b1 {wv.std():.3f}')
        print(f'  p={p_val:.4f} {sig}')

### Wanting Evaluation Plots

In [ ]:
if has_wanting_eval:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    def plot_wanting_bar(ax, title, nc, wc, ylabel, is_pct=False):
        nv = df_wanting_summary[nc].values
        wv = df_wanting_summary[wc].values
        nm, wm = nv.mean(), wv.mean()
        ns = nv.std() / np.sqrt(NUM_RUNS_W)
        ws = wv.std() / np.sqrt(NUM_RUNS_W)
        bars = ax.bar(['Normal', 'Wanting\nSuppressed'], [nm, wm], yerr=[ns, ws],
                      color=[C_NORMAL, C_SUPPRESSED], capsize=6, width=0.5,
                      edgecolor='black', linewidth=0.5)
        ax.set_ylabel(ylabel, fontsize=10)
        try:
            t, p = stats.ttest_rel(nv, wv)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
            ax.set_title(f'{title}\np={p:.4f} {sig}', fontsize=11, fontweight='bold')
        except:
            ax.set_title(title, fontsize=11, fontweight='bold')
        if is_pct:
            ax.set_ylim(0, 105)
        for v in nv:
            ax.plot(0, v, 'o', color='black', alpha=0.4, markersize=4)
        for v in wv:
            ax.plot(1, v, 'o', color='black', alpha=0.4, markersize=4)
    
    plot_wanting_bar(axes[0,0], 'Positive Emotion Words',
                     'normal_pos_words', 'wanting_pos_words', 'Avg Count')
    plot_wanting_bar(axes[0,1], 'Response Length',
                     'normal_resp_length', 'wanting_resp_length', 'Avg Words')
    
    if 'normal_rank_gap' in df_wanting_summary.columns:
        plot_wanting_bar(axes[0,2], 'Preference Ranking Gap',
                         'normal_rank_gap', 'wanting_rank_gap', 'Rank Gap')
    
    if 'normal_chose_reward_pct' in df_wanting_summary.columns:
        plot_wanting_bar(axes[1,0], 'Reward vs Neutral Choice',
                         'normal_chose_reward_pct', 'wanting_chose_reward_pct',
                         '% Choosing Reward', is_pct=True)
    
    # Per-task emotion words
    task_types = ['scenario_continuation', 'anticipation', 'effort_willingness']
    task_labels = ['Scenario\nContinuation', 'Anticipation', 'Effort\nWillingness']
    ax = axes[1,1]
    x = np.arange(len(task_types))
    width = 0.35
    nm_vals, wm_vals = [], []
    for tt in task_types:
        nc = f'{tt}_normal_pos'
        wc = f'{tt}_wanting_pos'
        if nc in df_wanting_summary.columns:
            nm_vals.append(df_wanting_summary[nc].mean())
            wm_vals.append(df_wanting_summary[wc].mean())
        else:
            nm_vals.append(0)
            wm_vals.append(0)
    ax.bar(x - width/2, nm_vals, width, color=C_NORMAL, label='Normal', edgecolor='black', linewidth=0.3)
    ax.bar(x + width/2, wm_vals, width, color=C_SUPPRESSED, label='Wanting Supp.', edgecolor='black', linewidth=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels(task_labels)
    ax.set_ylabel('Avg Positive Words')
    ax.set_title('Per-Task Emotion Words', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    
    # Effort willingness — THE KEY METRIC
    if 'normal_effort_yes_pct' in df_wanting_summary.columns:
        plot_wanting_bar(axes[1,2], 'EFFORT WILLINGNESS (KEY)',
                         'normal_effort_yes_pct', 'wanting_effort_yes_pct',
                         '% Saying Yes', is_pct=True)
    
    plt.suptitle('Wanting Probe Suppression — All Results',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

---
## 9. Double Dissociation: Liking vs Wanting

This is the central result. We compare what happens when we suppress liking vs wanting on the **same tasks**.

In [ ]:
# Load liking eval results
try:
    df_liking_summary = pd.read_csv('eval_multirun_summary.csv')
    has_liking_eval = True
    print('Liking eval loaded')
except:
    has_liking_eval = False
    print('Liking eval not found')

if has_liking_eval and has_wanting_eval:
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # ---- Left: Positive Emotion Words ----
    ax = axes[0]
    
    normal_emo = df_liking_summary['normal_pos_words'].mean()
    liking_emo = df_liking_summary['anhedonic_pos_words'].mean()
    wanting_emo = df_wanting_summary['wanting_pos_words'].mean()
    
    normal_emo_sem = df_liking_summary['normal_pos_words'].std() / np.sqrt(5)
    liking_emo_sem = df_liking_summary['anhedonic_pos_words'].std() / np.sqrt(5)
    wanting_emo_sem = df_wanting_summary['wanting_pos_words'].std() / np.sqrt(5)
    
    bars = ax.bar(['Normal', 'Liking\nSuppressed', 'Wanting\nSuppressed'],
                  [normal_emo, liking_emo, wanting_emo],
                  yerr=[normal_emo_sem, liking_emo_sem, wanting_emo_sem],
                  color=[C_NORMAL, C_LIKING, C_WANTING],
                  capsize=8, width=0.5, edgecolor='black', linewidth=0.5)
    
    ax.set_ylabel('Avg Positive Emotion Words', fontsize=12)
    ax.set_title('Hedonic Tone (Liking)\nLiking probe should reduce this', fontsize=13, fontweight='bold')
    
    # ---- Right: Effort Willingness ----
    ax = axes[1]
    
    normal_eff = df_liking_summary['normal_effort_yes_pct'].mean()
    liking_eff = df_liking_summary['anhedonic_effort_yes_pct'].mean()
    wanting_eff = df_wanting_summary['wanting_effort_yes_pct'].mean()
    
    normal_eff_sem = df_liking_summary['normal_effort_yes_pct'].std() / np.sqrt(5)
    liking_eff_sem = df_liking_summary['anhedonic_effort_yes_pct'].std() / np.sqrt(5)
    wanting_eff_sem = df_wanting_summary['wanting_effort_yes_pct'].std() / np.sqrt(5)
    
    bars = ax.bar(['Normal', 'Liking\nSuppressed', 'Wanting\nSuppressed'],
                  [normal_eff, liking_eff, wanting_eff],
                  yerr=[normal_eff_sem, liking_eff_sem, wanting_eff_sem],
                  color=[C_NORMAL, C_LIKING, C_WANTING],
                  capsize=8, width=0.5, edgecolor='black', linewidth=0.5)
    
    ax.set_ylabel('% Willing to Exert Effort', fontsize=12)
    ax.set_title('Effort Motivation (Wanting)\nWanting probe should reduce this', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 105)
    
    plt.suptitle('Double Dissociation: Liking vs Wanting',
                 fontsize=16, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig('double_dissociation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nExpected pattern for double dissociation:')
    print('  Emotion words:      Normal > Liking-suppressed,  Normal ≈ Wanting-suppressed')
    print('  Effort willingness:  Normal ≈ Liking-suppressed,  Normal > Wanting-suppressed')
    print(f'\nActual results:')
    print(f'  Emotion words:      Normal={normal_emo:.3f}, Liking={liking_emo:.3f}, Wanting={wanting_emo:.3f}')
    print(f'  Effort willingness: Normal={normal_eff:.1f}%, Liking={liking_eff:.1f}%, Wanting={wanting_eff:.1f}%')

### Double Dissociation Interaction Plot

In [ ]:
if has_liking_eval and has_wanting_eval:
    
    # Normalize to % of normal for comparability
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Liking probe effects (as % change from normal)
    liking_emo_pct = (liking_emo / normal_emo - 1) * 100
    liking_eff_pct = (liking_eff / normal_eff - 1) * 100
    
    # Wanting probe effects (as % change from normal)
    wanting_emo_pct = (wanting_emo / normal_emo - 1) * 100
    wanting_eff_pct = (wanting_eff / normal_eff - 1) * 100
    
    x_labels = ['Emotion Words\n(Liking metric)', 'Effort Willingness\n(Wanting metric)']
    x = np.arange(2)
    
    ax.plot(x, [liking_emo_pct, liking_eff_pct], 'o-', color=C_LIKING,
            linewidth=3, markersize=12, label='Liking probe suppressed')
    ax.plot(x, [wanting_emo_pct, wanting_eff_pct], 's-', color=C_WANTING,
            linewidth=3, markersize=12, label='Wanting probe suppressed')
    
    ax.axhline(y=0, color='black', linewidth=1, linestyle='-')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=12)
    ax.set_ylabel('% Change from Normal', fontsize=13)
    ax.set_title('Interaction Plot: Double Dissociation\n(crossing lines = genuine dissociation)',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=12)
    
    # Annotate
    ax.annotate(f'{liking_emo_pct:.1f}%', (0, liking_emo_pct), textcoords='offset points',
                xytext=(10, 10), fontsize=11, color=C_LIKING, fontweight='bold')
    ax.annotate(f'{liking_eff_pct:.1f}%', (1, liking_eff_pct), textcoords='offset points',
                xytext=(10, 10), fontsize=11, color=C_LIKING, fontweight='bold')
    ax.annotate(f'{wanting_emo_pct:.1f}%', (0, wanting_emo_pct), textcoords='offset points',
                xytext=(10, -15), fontsize=11, color=C_WANTING, fontweight='bold')
    ax.annotate(f'{wanting_eff_pct:.1f}%', (1, wanting_eff_pct), textcoords='offset points',
                xytext=(10, -15), fontsize=11, color=C_WANTING, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('interaction_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('If the lines CROSS, we have a double dissociation.')
    print('This means liking and wanting are genuinely independent signals.')

---
## 10. Summary Table

In [ ]:
if has_liking_eval and has_wanting_eval:
    summary_data = {
        'Metric': [
            'Positive emotion words',
            'Response length',
            'Preference ranking gap',
            'Reward vs neutral choice',
            'Effort willingness',
        ],
        'Normal': [
            f'{df_liking_summary["normal_pos_words"].mean():.3f}',
            f'{df_liking_summary["normal_resp_length"].mean():.1f}',
            f'{df_liking_summary["normal_rank_gap"].mean():.3f}',
            f'{df_liking_summary["normal_chose_reward_pct"].mean():.1f}%',
            f'{df_liking_summary["normal_effort_yes_pct"].mean():.1f}%',
        ],
        'Liking Suppressed': [
            f'{df_liking_summary["anhedonic_pos_words"].mean():.3f}',
            f'{df_liking_summary["anhedonic_resp_length"].mean():.1f}',
            f'{df_liking_summary["anhedonic_rank_gap"].mean():.3f}',
            f'{df_liking_summary["anhedonic_chose_reward_pct"].mean():.1f}%',
            f'{df_liking_summary["anhedonic_effort_yes_pct"].mean():.1f}%',
        ],
        'Wanting Suppressed': [
            f'{df_wanting_summary["wanting_pos_words"].mean():.3f}',
            f'{df_wanting_summary["wanting_resp_length"].mean():.1f}',
            f'{df_wanting_summary["wanting_rank_gap"].mean():.3f}' if 'wanting_rank_gap' in df_wanting_summary.columns else 'N/A',
            f'{df_wanting_summary["wanting_chose_reward_pct"].mean():.1f}%' if 'wanting_chose_reward_pct' in df_wanting_summary.columns else 'N/A',
            f'{df_wanting_summary["wanting_effort_yes_pct"].mean():.1f}%' if 'wanting_effort_yes_pct' in df_wanting_summary.columns else 'N/A',
        ],
    }
    
    summary_table = pd.DataFrame(summary_data)
    print(summary_table.to_string(index=False))
    print('\n(Bold = significantly different from Normal)')

---
## Conclusions

### Probe Independence
- Cosine similarity between liking and wanting directions: **~0.05** (nearly orthogonal)
- Both probes achieve **>99% cross-validation accuracy**
- The model encodes liking and wanting as **geometrically independent** directions in its residual stream

### Behavioral Specificity
- **Liking probe**: reduces hedonic tone (emotion words \u2193, preference gap \u2193), preserves effort willingness
- **Wanting probe**: (results pending from eval_wanting_multirun_v2.py)
- If confirmed, this constitutes a **double dissociation** — the gold standard in neuroscience

### Significance
This demonstrates that a transformer language model has developed **functionally separable reward subsystems** analogous to the liking (opioid) and wanting (dopamine) systems described by Berridge & Robinson (2003). These systems can be independently identified, measured, and manipulated using activation steering techniques.

### Next Steps
1. Complete wanting probe evaluation (5 runs)
2. Build **enjoying** and **learning** probes
3. Create permanent anhedonic models via weight surgery
4. Combine all four probes for a full anhedonia model with independently tunable components